In [10]:

############################################
# CS 340 Project Two
# Grazioso Salvare Rescue Animal Dashboard
# Created by Arthur Sarkisyan
############################################

# Imports
import os
import base64
import logging
import getpass

import pandas as pd
import plotly.express as px
import dash_leaflet as dl

from jupyter_dash import JupyterDash
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output

from CRUD_Python_Module import AnimalShelter


############################################
# Logging Configuration
############################################

logging.getLogger("werkzeug").setLevel(logging.ERROR)

JupyterDash.infer_jupyter_proxy_config()


############################################
# MongoDB Connection
############################################

username = "aacuser"

password = os.environ.get("AAC_PASSWORD")

if not password:
    password = getpass.getpass(
        "Enter MongoDB password: "
    )

    if not password:
        raise RuntimeError(
            "A MongoDB password is required."
        )

    os.environ["AAC_PASSWORD"] = password

print("Password configured:", bool(password))

db = AnimalShelter(username, password)

df = pd.DataFrame.from_records(db.read({}))

original_count = len(df)

df.drop(
    columns=["_id"],
    inplace=True,
    errors="ignore"
)

df.drop_duplicates(inplace=True)

df.reset_index(drop=True, inplace=True)

print("Database connection successful!")
print("Original MongoDB records:", original_count)
print("Unique dashboard records:", len(df))
print("Duplicates removed:", original_count - len(df))


############################################
# Rescue Filter Queries
############################################

rescue_queries = {

    "Water Rescue": {
        "animal_type": "Dog",
        "breed": {
            "$in": [
                "Labrador Retriever Mix",
                "Chesapeake Bay Retriever",
                "Newfoundland"
            ]
        },
        "sex_upon_outcome": "Intact Female",
        "age_upon_outcome_in_weeks": {
            "$gte": 26,
            "$lte": 156
        }
    },

    "Mountain or Wilderness Rescue": {
        "animal_type": "Dog",
        "breed": {
            "$in": [
                "German Shepherd",
                "Alaskan Malamute",
                "Old English Sheepdog",
                "Siberian Husky",
                "Rottweiler"
            ]
        },
        "sex_upon_outcome": "Intact Male",
        "age_upon_outcome_in_weeks": {
            "$gte": 26,
            "$lte": 156
        }
    },

    "Disaster or Individual Tracking": {
        "animal_type": "Dog",
        "breed": {
            "$in": [
                "Doberman Pinscher",
                "German Shepherd",
                "Golden Retriever",
                "Bloodhound",
                "Rottweiler"
            ]
        },
        "sex_upon_outcome": "Intact Male",
        "age_upon_outcome_in_weeks": {
            "$gte": 20,
            "$lte": 300
        }
    }
}


############################################
# Retrieve Filtered Records
############################################

def get_rescue_data(filter_type):

    if filter_type == "Reset":
        return df.to_dict("records")

    query = rescue_queries.get(filter_type)

    if query is None:
        return df.to_dict("records")

    records = db.read(query)

    filtered_df = pd.DataFrame.from_records(
        records
    )

    if filtered_df.empty:
        return []

    filtered_df.drop(
        columns=["_id"],
        inplace=True,
        errors="ignore"
    )

    filtered_df.drop_duplicates(inplace=True)

    filtered_df.reset_index(
        drop=True,
        inplace=True
    )

    return filtered_df.to_dict("records")


############################################
# Dashboard Application
############################################

app = JupyterDash(__name__)

image_filename = "Grazioso Salvare Logo.png"

with open(image_filename, "rb") as image_file:
    encoded_image = base64.b64encode(
        image_file.read()
    ).decode("utf-8")


############################################
# Dashboard Layout
############################################

app.layout = html.Div([

    html.Center([

        html.Img(
            src="data:image/png;base64,{}".format(
                encoded_image
            ),
            style={
                "width": "200px",
                "height": "auto"
            }
        ),

        html.H1(
            "Grazioso Salvare Rescue Animal Dashboard"
        ),

        html.H3(
            "Created by Arthur Sarkisyan"
        )

    ]),

    html.Hr(),

    html.H3("Rescue Type Filters"),

    dcc.RadioItems(
        id="filter-type",
        options=[
            {
                "label": "Water Rescue",
                "value": "Water Rescue"
            },
            {
                "label": "Mountain or Wilderness Rescue",
                "value": "Mountain or Wilderness Rescue"
            },
            {
                "label": "Disaster or Individual Tracking",
                "value": "Disaster or Individual Tracking"
            },
            {
                "label": "Reset",
                "value": "Reset"
            }
        ],
        value="Reset",
        labelStyle={
            "display": "inline-block",
            "marginRight": "25px"
        }
    ),

    html.Hr(),

    dash_table.DataTable(

        id="datatable-id",

        columns=[
            {
                "name": i,
                "id": i,
                "deletable": False,
                "selectable": True
            }
            for i in df.columns
        ],

        data=df.to_dict("records"),

        editable=False,

        filter_action="native",

        sort_action="native",

        sort_mode="multi",

        column_selectable="single",

        row_selectable="single",

        selected_rows=[0],

        page_action="native",

        page_current=0,

        page_size=10,

        style_table={
            "overflowX": "auto"
        },

        style_cell={
            "textAlign": "left",
            "padding": "8px",
            "fontFamily": "Arial",
            "fontSize": "12px",
            "minWidth": "100px",
            "maxWidth": "220px",
            "overflow": "hidden",
            "textOverflow": "ellipsis"
        },

        style_header={
            "backgroundColor": "#f2f2f2",
            "fontWeight": "bold"
        }

    ),

    html.Br(),

    html.Hr(),

    html.Div(

        style={
            "display": "flex",
            "flexWrap": "wrap",
            "gap": "15px"
        },

        children=[

            html.Div(
                id="graph-id",
                style={
                    "flex": "1 1 450px",
                    "minWidth": "0"
                }
            ),

            html.Div(
                id="map-id",
                style={
                    "flex": "1 1 450px",
                    "minWidth": "0"
                }
            )

        ]
    )

])


############################################
# Dashboard Filtering Callback
############################################

@app.callback(
    Output("datatable-id", "data"),
    Input("filter-type", "value")
)
def update_dashboard(filter_type):

    return get_rescue_data(filter_type)


############################################
# Reset Table Selection
############################################

@app.callback(
    Output("datatable-id", "selected_rows"),
    Output("datatable-id", "page_current"),
    Input("filter-type", "value")
)
def reset_table_selection(filter_type):

    return [0], 0


############################################
# Interactive Breed Distribution Chart
############################################

@app.callback(
    Output("graph-id", "children"),
    Input("datatable-id", "derived_virtual_data")
)
def update_graphs(viewData):

    if viewData is None:
        viewData = df.to_dict("records")

    dff = pd.DataFrame.from_records(viewData)

    if dff.empty or "breed" not in dff.columns:
        return html.P(
            "No animal records available."
        )

    breed_counts = dff["breed"].value_counts()

    top_breeds = breed_counts.head(10)

    other_count = breed_counts.iloc[10:].sum()

    chart_data = pd.DataFrame({
        "Breed": top_breeds.index,
        "Count": top_breeds.values
    })

    if other_count > 0:

        chart_data = pd.concat(
            [
                chart_data,
                pd.DataFrame({
                    "Breed": ["Other"],
                    "Count": [other_count]
                })
            ],
            ignore_index=True
        )

    fig = px.pie(
        chart_data,
        names="Breed",
        values="Count",
        title="Animal Breed Distribution (Top 10 Breeds)"
    )

    fig.update_traces(
        textposition="inside",
        textinfo="percent"
    )

    fig.update_layout(
        height=500,
        margin=dict(
            l=20,
            r=20,
            t=60,
            b=20
        )
    )

    return dcc.Graph(
        figure=fig,
        config={
            "displayModeBar": False
        }
    )


############################################
# Highlight Selected Columns
############################################

@app.callback(
    Output(
        "datatable-id",
        "style_data_conditional"
    ),
    Input(
        "datatable-id",
        "selected_columns"
    )
)
def update_styles(selected_columns):

    if selected_columns is None:
        selected_columns = []

    return [
        {
            "if": {
                "column_id": column
            },
            "backgroundColor": "#D2F3FF"
        }
        for column in selected_columns
    ]


############################################
# Interactive Geolocation Map
############################################

@app.callback(
    Output("map-id", "children"),
    Input(
        "datatable-id",
        "derived_virtual_data"
    ),
    Input(
        "datatable-id",
        "derived_virtual_selected_rows"
    )
)
def update_map(viewData, index):

    if viewData is None:
        viewData = df.to_dict("records")

    dff = pd.DataFrame.from_records(viewData)

    if dff.empty:
        return html.P(
            "No location data available."
        )

    if not index:
        row = 0
    else:
        row = index[0]

    if row >= len(dff):
        row = 0

    selected_animal = dff.iloc[row]

    latitude = pd.to_numeric(
        selected_animal.get("location_lat"),
        errors="coerce"
    )

    longitude = pd.to_numeric(
        selected_animal.get("location_long"),
        errors="coerce"
    )

    if (
        pd.isna(latitude)
        or pd.isna(longitude)
        or not (-90 <= latitude <= 90)
        or not (-180 <= longitude <= 180)
    ):
        return html.P(
            "Valid location data is unavailable "
            "for this animal."
        )

    breed = str(
        selected_animal.get(
            "breed",
            "Unknown"
        )
    )

    animal_name = str(
        selected_animal.get(
            "name",
            "Unknown"
        )
    )

    if not animal_name.strip():
        animal_name = "Unknown"

    return dl.Map(

        style={
            "width": "100%",
            "height": "500px"
        },

        center=[
            float(latitude),
            float(longitude)
        ],

        zoom=10,

        children=[

            dl.TileLayer(),

            dl.Marker(

                position=[
                    float(latitude),
                    float(longitude)
                ],

                children=[

                    dl.Tooltip(breed),

                    dl.Popup([

                        html.H3(
                            "Animal Information"
                        ),

                        html.P(
                            "Name: " + animal_name
                        ),

                        html.P(
                            "Breed: " + breed
                        )

                    ])

                ]
            )

        ]
    )


############################################
# Run Dashboard
############################################

logging.getLogger("werkzeug").setLevel(logging.ERROR)

print()
print("Grazioso Salvare Dashboard")
print("Dashboard is ready!")
print()

app.run_server(
    mode="external",
    host="0.0.0.0",
    port=8050,
    debug=False
)

Enter MongoDB password:  ········


Password configured: True
Database connection successful!
Original MongoDB records: 20000
Unique dashboard records: 10000
Duplicates removed: 10000

Grazioso Salvare Dashboard
Dashboard is ready!

Dash app running on https://majordesign-contactspecial-3000.codio.io/proxy/8050/
